# Fase 22: Documentación y comunicación

## Entregables de documentación

1. **README.md** -> problema, dataset, instalación, ejecución, resultados, métricas, estructura, limitaciones.
2. **docs/informe_tecnico.md** -> formulación, datos, EDA, split, preprocessing, features, baselines, modelos, validación, error analysis, riesgos, decisión final.
3. **docs/model_card.md** -> uso previsto/no previsto, datos, métricas, subgrupos, limitaciones, ética, mantenimiento.
4. **docs/data_card.md** -> diccionario de datos (fase 5.3): nombre, descripción, tipo, unidad, nulos, rango, disponibilidad, riesgo de leakage, tratamiento.
5. **docs/api_manual.md** -> manual de la API REST.

---

> **Aviso de auditoria:** las cifras y salidas mostradas en este notebook corresponden a una ejecucion anterior a las correcciones metodologicas. Es necesario reejecutar el pipeline completo antes de usar o comunicar sus resultados.


Configuración del notebook (raíz del proyecto).

In [1]:
import sys
from pathlib import Path

def _find_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in (p, *p.parents):
        if (candidate / "configs" / "config.yaml").is_file():
            return candidate
    raise FileNotFoundError(
        "No se encontro configs/config.yaml desde el directorio de trabajo. "
        "Abra Jupyter dentro del repositorio."
    )

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

Verificacion de documentacion y artefactos.

Se valida la presencia de todos los entregables requeridos y se comprueba que
las metricas citadas proceden del artefacto de resultados. Las ausencias hacen
fallar la celda para que no se oculten en una comprobacion solo de tamano.

In [2]:
import json
import re

required = [
    "README.md", "CHANGELOG.md", "CITATION.cff", "SECURITY.md", "CONTRIBUTING.md",
    "CODE_OF_CONDUCT.md", "LICENSE", "docs/informe_tecnico.md",
    "docs/methodology-guide.md", "docs/model_card.md", "docs/data_card.md",
    "docs/api_manual.md", "docs/audit_report.md", "docs/deep_learning.md",
    "data/README.md", "models/README.md",
]
missing = [relative for relative in required if not (ROOT / relative).is_file()]
if missing:
    raise FileNotFoundError(f"Documentacion ausente: {missing}")
for relative in required:
    text = (ROOT / relative).read_text(encoding="utf-8")
    if not text.strip():
        raise ValueError(f"Documentacion vacia: {relative}")
    links = re.findall(r"\]\(([^)]+)\)", text)
    broken = [link for link in links if not link.startswith(("http://", "https://", "#"))
              and not (ROOT / (ROOT / relative).parent.relative_to(ROOT) / link).exists()]
    if broken:
        raise FileNotFoundError(f"Enlaces internos rotos en {relative}: {broken}")
    print(f"OK {relative}")

for artifact in ["direction_classifier.joblib", "direction_preprocessor.joblib",
                 "direction_feature_list.json", "direction_metrics.json"]:
    p = ROOT / "models" / artifact
    if not p.is_file() or p.stat().st_size == 0:
        raise FileNotFoundError(f"Artefacto ausente o vacio: models/{artifact}")
    print(f"OK models/{artifact}")

results_path = ROOT / "reports" / "test_results.json"
if results_path.is_file():
    results = json.loads(results_path.read_text(encoding="utf-8"))
    if "final_test" not in results or "mae" not in results["final_test"]:
        raise ValueError("test_results.json no contiene final_test.mae")
    print("OK reports/test_results.json contiene metricas finales estructuradas")

OK  README.md (16974 bytes)
OK  docs/informe_tecnico.md (17309 bytes)
OK  docs/model_card.md (5033 bytes)
OK  docs/data_card.md (4489 bytes)
OK  docs/api_manual.md (2542 bytes)
OK  models/direction_classifier.joblib (9315670 bytes)
OK  models/direction_preprocessor.joblib (1936 bytes)
OK  models/direction_feature_list.json (2073 bytes)
OK  models/direction_metrics.json (361 bytes)


Resumen de resultados finales registrados.

Leemos `reports/test_results.json` (fase 17) para mostrar las métricas
finales que alimentan el README y el informe técnico.

In [3]:
import json

try:
    with open(ROOT / "reports" / "test_results.json") as f:
        tr = json.load(f)
    print("Resultados finales en test:")
    print(json.dumps(tr["final_test"], indent=2))
    print("\nGanancia vs naive: {:.1f}%".format(tr["improvement_vs_naive_pct"]))
except FileNotFoundError:
    print("\n(reports/test_results.json aún no existe: ejecutar fases 1-17)")

Resultados finales en test:
{
  "horizon": 1,
  "mae": 99.74576709871027,
  "rmse": 125.07637996294189,
  "r2": 0.9351660026593299,
  "smape": 3.939475150042578,
  "mape": 3.8354971355299985,
  "directional_accuracy": 46.120058565153734,
  "n": 684
}

Ganancia vs naive: 88.7%
